# Full Bering Sea: per-pixel MODIS-Aqua a$_{dg}$(443) & K$_d$(490) trends (2002–present)

Companion to `09_bering-and-anadyr-caches.ipynb` — same data (MODIS-Aqua L3-mapped, 8-day, 4 km;
`MODISA_L3m_IOP` adg_443 and `MODISA_L3m_KD` Kd_490), same trend recipe
(Theil–Sen slope + AR(1)-corrected Mann–Kendall per pixel, from
`the DBO weekly-pipeline trend recipe (not included)`), but applied to **every pixel of the whole
Bering Sea** instead of the three boxes. The boxes A/B/C are drawn on the maps for context.

Differences from the box notebook worth knowing:
* **The domain crosses the antimeridian** (the Russian side is in °E). The L3m grid runs −180…180°,
  so each granule is subset in two pieces and stitched onto a continuous **0–360° longitude axis**
  (162°E … 203.5°E, i.e. 156.5°W).
* **Its own cache** (`bering_full_cache/`, ~2–3 GB; the first streaming pass is a few hours). The box
  notebook's cache is a spatial subset of this one, so once this cache exists you could point the box
  notebook at it — but they're kept separate so neither invalidates the other.
* Annual May–Sep median composites are built **year by year from the cache** (never loading the full
  ~1100-composite cube into memory) and saved to netCDF, so the expensive parts only ever run once.
* The per-pixel fit covers ~250k ocean pixels — expect the trend loop to take roughly 5–15 min per
  product, with progress printed.

Runs in the `pace_chl_v2` environment from `environment.yaml`.

In [1]:
import warnings, time as _time
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import theilslopes, kendalltau, norm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import earthaccess

warnings.filterwarnings("ignore", category=RuntimeWarning)
xr.set_options(keep_attrs=True);

## 1. Configuration

`DOMAIN` is the full Bering Sea: Aleutians to Bering Strait, Kamchatka to Alaska. Longitudes are
stored 0–360 (162 → 203.5) so the field is continuous across the dateline; the two `lon_slices`
are what get cut from each −180…180 granule before stitching.

In [2]:
# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
DOMAIN = dict(
    lat=(51.5, 66.5),                       # Aleutian arc -> Bering Strait
    lon360=(162.0, 203.5),                  # 162E -> 156.5W, on a 0-360 axis
    lon_slices=[(162.0, 180.0), (-180.0, -156.5)],   # how that maps onto the -180..180 grid
)

# The three study boxes from the box notebook — overlay only (edit to match that notebook!)
BOXES = {
    "A": dict(lat=(57.7, 59.7), lon=(-171.5, -166.5), color="#0072B2"),
    "B": dict(lat=(60.2, 62.2), lon=(-170.5, -165.5), color="#E69F00"),
    "C": dict(lat=(62.4, 64.4), lon=(-169.0, -164.0), color="#009E73"),
}
def to360(lon):
    return lon + 360.0 if lon < 0 else lon

# River outflow markers (lon in -180..180; converted with to360 at plot time)
RIVERS = {
    "Yukon R.":     dict(lat=62.85, lon=-164.70),   # delta front, main (south) mouths
    "Kuskokwim R.": dict(lat=59.90, lon=-162.45),   # into Kuskokwim Bay
    "Anadyr R.":    dict(lat=64.75, lon=177.55),    # Russian side, Gulf of Anadyr
}

# DBO 2 (Chirikov Basin) — canonical polygon from the DBO weekly-pipeline box definitions (not included)
DBO2_RING = [(65.107747, -170.865176), (65.097001, -167.859549),
             (64.481828, -167.907758), (64.493658, -170.840669)]   # (lat, lon) corners
DBO2_COLOR = "#CC79A7"

def ring_xy(ring):
    """Close the polygon and return plot-ready (lon360 list, lat list)."""
    lats = [q[0] for q in ring] + [ring[0][0]]
    lons = [to360(q[1]) for q in ring] + [to360(ring[0][1])]
    return lons, lats

PRODUCTS = {
    "adg443": dict(short_name="MODISA_L3m_IOP", pattern="*.8D.IOP.adg_443.4km.nc",
                   var_candidates=["adg_443", "adg_443_giop"], name="adg_443 (GIOP)"),
    "kd490":  dict(short_name="MODISA_L3m_KD",  pattern="*.8D.KD.Kd_490.4km.nc",
                   var_candidates=["Kd_490"], name="Kd_490"),
}

START_DATE = date(2002, 7, 1)
END_DATE   = date.today()

SEASON_MONTHS = (5, 6, 7, 8, 9)     # open-water season for the annual composites
MIN_COMPOSITES_PER_YEAR = 6
MIN_YEARS_FOR_TREND = 10

CACHE  = Path("bering_full_cache")
OUTDIR = Path("bering_full_output")
for p in (CACHE, OUTDIR):
    p.mkdir(exist_ok=True)

## 3. Find the granules

In [5]:
auth = earthaccess.login(persist=True)

In [6]:
def _search(short_name, pattern, s, e, tries=5, limit=2000):
    for k in range(tries):
        try:
            return earthaccess.search_data(short_name=short_name, granule_name=pattern,
                                           temporal=(s, e), count=limit)
        except Exception as ex:
            print("  retry:", str(ex)[:80]); _time.sleep(2 * (k + 1))
    raise RuntimeError(f"CMR search failed for {short_name}")

def granule_fname(g):
    return g.data_links()[0].rsplit("/", 1)[-1]

def granule_times(fname):
    d0, d1 = fname.split(".")[1].split("_")
    t0, t1 = pd.Timestamp(d0), pd.Timestamp(d1)
    return t0, t1, t0 + (t1 - t0) / 2

granules = {}
for key, prod in PRODUCTS.items():
    results = []
    for yr in range(START_DATE.year, END_DATE.year + 1):
        s = max(START_DATE, date(yr, 1, 1))
        e = min(END_DATE, date(yr, 12, 31))
        results += _search(prod["short_name"], prod["pattern"], s.isoformat(), e.isoformat())
    seen, uniq = set(), []
    for g in results:
        f = granule_fname(g)
        if f not in seen:
            seen.add(f); uniq.append(g)
    granules[key] = sorted(uniq, key=granule_fname)
    print(f"{key}: {len(granules[key])} 8-day granules")

adg443: 1099 8-day granules
kd490: 1099 8-day granules


## 4. Stream + cache the Bering subsets

Each granule is cut to the two longitude slices, stitched onto the 0–360 axis (~360 × 1000 px,
≈1.5 MB) and cached. Interruptible/resumable exactly like the box notebook; failures get one retry
pass, and re-running the cell retries whatever is left.

In [7]:
def pick_var(ds, candidates):
    for v in candidates:
        if v in ds.data_vars:
            return v
    for v, da in ds.data_vars.items():
        if {"lat", "lon"} <= set(da.dims):
            return v
    raise KeyError(f"none of {candidates} in {list(ds.data_vars)}")

def subset_bering(ds):
    """Cut the two dateline-straddling slices and stitch onto a 0-360 longitude axis."""
    lat0, lat1 = DOMAIN["lat"]
    lat_desc = ds.lat.values[0] > ds.lat.values[-1]
    lat_sl = slice(lat1, lat0) if lat_desc else slice(lat0, lat1)
    parts = []
    for lo0, lo1 in DOMAIN["lon_slices"]:
        p = ds.sel(lat=lat_sl, lon=slice(lo0, lo1))
        if p.sizes.get("lon", 0):
            parts.append(p.assign_coords(lon=np.where(p.lon.values < 0,
                                                      p.lon.values + 360.0, p.lon.values)))
    return xr.concat(parts, dim="lon").sortby("lon")

def cache_granule(g, key, prod):
    fname = granule_fname(g)
    out = CACHE / key / fname
    if out.exists():
        return "cached"
    f = earthaccess.open([g], show_progress=False)[0]
    try:
        with xr.open_dataset(f, engine="h5netcdf") as ds:
            v = pick_var(ds, prod["var_candidates"])
            sub = subset_bering(ds[[v]]).load()
    finally:
        f.close()
    t0, t1, tm = granule_times(fname)
    sub = sub.rename({v: key}).expand_dims(time=[tm])
    sub[key].attrs.update(source_file=fname, composite_window=f"{t0:%Y%m%d}-{t1:%Y%m%d}",
                          lon_convention="0-360 (continuous across the dateline)")
    sub.to_netcdf(out)
    return "new"

for key, prod in PRODUCTS.items():
    (CACHE / key).mkdir(exist_ok=True)
    gs = granules[key]
    print(f"--- {key}: {len(gs)} granules ---")
    t0 = _time.time(); new = 0; failures = []
    for i, g in enumerate(gs):
        try:
            if cache_granule(g, key, prod) == "new":
                new += 1
        except Exception as ex:
            failures.append(g)
            print(f"  FAIL {granule_fname(g)}: {str(ex)[:70]}")
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(gs)}  ({new} new, {len(failures)} failed, {(_time.time()-t0)/60:.1f} min)")
    still = []
    for g in failures:
        try:
            cache_granule(g, key, prod)
        except Exception:
            still.append(granule_fname(g))
    print(f"done: {new} new, {len(still)} still failing after retry, {(_time.time()-t0)/60:.1f} min")
    if still:
        print("  re-run this cell to retry:", still[:5], "..." if len(still) > 5 else "")

--- adg443: 1099 granules ---
  100/1099  (100 new, 0 failed, 0.6 min)
  200/1099  (200 new, 0 failed, 1.2 min)
  300/1099  (300 new, 0 failed, 1.7 min)
  400/1099  (400 new, 0 failed, 2.3 min)
  500/1099  (500 new, 0 failed, 2.8 min)
  600/1099  (600 new, 0 failed, 3.3 min)
  700/1099  (700 new, 0 failed, 3.9 min)
  800/1099  (800 new, 0 failed, 4.4 min)
  900/1099  (900 new, 0 failed, 5.0 min)
  1000/1099  (1000 new, 0 failed, 5.5 min)
done: 1099 new, 0 still failing after retry, 6.0 min
--- kd490: 1099 granules ---
  100/1099  (100 new, 0 failed, 0.6 min)
  200/1099  (200 new, 0 failed, 1.1 min)
  300/1099  (300 new, 0 failed, 1.7 min)
  400/1099  (400 new, 0 failed, 2.2 min)
  500/1099  (500 new, 0 failed, 2.8 min)
  600/1099  (600 new, 0 failed, 3.3 min)
  700/1099  (700 new, 0 failed, 3.9 min)
  800/1099  (800 new, 0 failed, 4.5 min)
  900/1099  (900 new, 0 failed, 5.1 min)
  1000/1099  (1000 new, 0 failed, 5.6 min)
done: 1099 new, 0 still failing after retry, 6.0 min


## 5. Annual May–Sep median composites, built year by year

Same logic as the box notebook (one May–Sep median per pixel per year, masked where a pixel-year has
fewer than `MIN_COMPOSITES_PER_YEAR` 8-day values) — but computed one year at a time from the cache
so memory stays small, and saved to `bering_full_output/annual_maysep_<product>.nc`. Delete that file
to force a rebuild (e.g. after changing `SEASON_MONTHS`).

In [8]:
def annual_from_cache(key):
    out = OUTDIR / f"annual_maysep_{key}.nc"
    if out.exists():
        print(f"{key}: loading existing {out.name}")
        return xr.open_dataarray(out)
    byyr = {}
    for f in sorted((CACHE / key).glob("*.nc")):
        tm = granule_times(f.name)[2]
        if tm.month in SEASON_MONTHS:
            byyr.setdefault(tm.year, []).append(f)
    fields, yrs = [], sorted(byyr)
    for yr in yrs:
        ds = xr.open_mfdataset(byyr[yr], combine="nested", concat_dim="time", engine="h5netcdf",
                               data_vars="minimal", coords="minimal", compat="override")
        da = ds[key].load(); ds.close()
        med = da.median("time")
        cnt = np.isfinite(da).sum("time")
        fields.append(med.where(cnt >= MIN_COMPOSITES_PER_YEAR))
        print(f"  {key} {yr}: {len(byyr[yr])} May–Sep composites")
    ann = xr.concat(fields, dim=pd.Index(yrs, name="year"))
    ann.name = key
    ann.to_netcdf(out)
    return ann

annual = {key: annual_from_cache(key) for key in PRODUCTS}
for key, a in annual.items():
    print(key, dict(a.sizes))

  adg443 2002: 11 May–Sep composites
  adg443 2003: 19 May–Sep composites
  adg443 2004: 19 May–Sep composites
  adg443 2005: 19 May–Sep composites
  adg443 2006: 19 May–Sep composites
  adg443 2007: 19 May–Sep composites
  adg443 2008: 19 May–Sep composites
  adg443 2009: 19 May–Sep composites
  adg443 2010: 19 May–Sep composites
  adg443 2011: 19 May–Sep composites
  adg443 2012: 19 May–Sep composites
  adg443 2013: 19 May–Sep composites
  adg443 2014: 19 May–Sep composites
  adg443 2015: 19 May–Sep composites
  adg443 2016: 19 May–Sep composites
  adg443 2017: 19 May–Sep composites
  adg443 2018: 19 May–Sep composites
  adg443 2019: 19 May–Sep composites
  adg443 2020: 19 May–Sep composites
  adg443 2021: 19 May–Sep composites
  adg443 2022: 19 May–Sep composites
  adg443 2023: 19 May–Sep composites
  adg443 2024: 19 May–Sep composites
  adg443 2025: 19 May–Sep composites
  adg443 2026: 3 May–Sep composites
  kd490 2002: 11 May–Sep composites
  kd490 2003: 19 May–Sep composites
  kd

---

# Part B — OISSS salinity stack on the western (Anadyr) grid

*This part was a separate notebook before the repository was consolidated; it starts from its own imports and configuration, so it can be run on its own after the sections above.*

# OISSS Phase 2 — freshening on the Anadyr side, on the EASE-2 grid

The Yukon-side OISSS analysis lives on the EASE-2 `dbo_area` grid; this notebook keeps
**grid parity** for the Anadyr side. One catch: in that projection (LAEA, lon_0=0) east
longitudes project to positive x, and `dbo_area`'s extent ends at exactly x = 0 — the
dateline is the grid's right edge, which is why the gulf was never in the original stack.
So we extend the extent east by 100 cells (**same projection, same 4 km cells, first 906
columns identical to `dbo_area`** — every Yukon-side cell is unchanged), regrid OISSS
monthly SSS onto it with the same gauss/30 km recipe as the original stack, and run:

1. monthly-anomaly Theil–Sen + Mann-Kendall (AR1) trends for the six regions,
2. June–October pixel trend maps (hatched p<0.05) with the boxes overlaid,
3. optional SSS ↔ MODIS a_dg anomaly correlation (needs the CSV exported from
   anadyr-side-phase1 — exact export cell given at §4).

Discriminator: **is freshening amplified on the Anadyr branch, matching the browning
gradient?** OISSS is served openly by ESR (no login). First build ~10 min.

In [4]:
import io, re, time as _time, urllib.request, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pyresample import AreaDefinition, geometry
from pyresample.kd_tree import resample_gauss
from scipy.stats import theilslopes, kendalltau, norm, t as tdist

warnings.filterwarnings("ignore", category=RuntimeWarning)
OISSS_MONTHLY = "https://oisss.esr.org/data/V3/Monthly"
SSS_VALID = (0.0, 45.0)
RESAMPLE, ROI = "gauss", 30_000            # verbatim from the Yukon-side OISSS builder
OUT = Path("oisss_west_cache"); OUT.mkdir(exist_ok=True)
FIGDIR = Path("tracing_output"); FIGDIR.mkdir(exist_ok=True)

# --- EASE-2: dbo_area extended 100 cells east (x: 0 -> +400 km). Cols 0-905 == dbo_area.
CELL = 4000.0
dbo_area_west = AreaDefinition(
    area_id="ease2_north_dbo_west_4km",
    description="dbo_area + 100-column eastward extension across the dateline",
    proj_id="EPSG:6931",
    projection={"proj": "laea", "lat_0": 90, "lon_0": 0, "x_0": 0, "y_0": 0,
                "datum": "WGS84", "units": "m"},
    width=1006, height=845, area_extent=(-3_624_000, 992_000, 400_000, 4_372_000))

REGIONS = {   # lat0, lat1, lon360_0, lon360_1, color — same boxes as the PACE/MODIS notebooks
    "AnadyrInner":  (63.6, 65.0, 176.5, 180.5, "#D55E00"),
    "AnadyrOuter":  (61.8, 63.6, 174.0, 179.5, "#56B4E9"),
    "AnadyrStrait": (63.3, 64.7, 186.5, 189.2, "#999933"),
    "DBO1":         (61.85, 63.77, 184.2, 187.8, "#AA4499"),
    "DBO2":         (64.48, 65.11, 189.1, 192.2, "#CC79A7"),
    "boxC":         (62.4, 64.4, 191.0, 196.0, "#009E73"),
}
OPEN_WATER_MONTHS = [6, 7, 8, 9, 10]

# masks on the full grid, then a cell-aligned crop covering all regions (+pad)
lons2d, lats2d = dbo_area_west.get_lonlats()
lon360_2d = lons2d % 360
masks_full = {n: ((lats2d >= a0) & (lats2d <= a1) & (lon360_2d >= o0) & (lon360_2d <= o1))
              for n, (a0, a1, o0, o1, _c) in REGIONS.items()}
union = np.zeros_like(lons2d, bool)
for m in masks_full.values():
    union |= m
jj, ii = np.where(union)
PAD = 15
j0, j1 = max(jj.min() - PAD, 0), min(jj.max() + PAD, dbo_area_west.height - 1)
i0, i1 = max(ii.min() - PAD, 0), min(ii.max() + PAD, dbo_area_west.width - 1)
xv, yv = dbo_area_west.get_proj_vectors()
work_area = AreaDefinition(
    area_id="ease2_west_work", description="cell-aligned crop", proj_id="EPSG:6931",
    projection=dbo_area_west.crs.to_dict() if hasattr(dbo_area_west, "crs")
               else dbo_area_west.proj_dict,
    width=i1 - i0 + 1, height=j1 - j0 + 1,
    area_extent=(xv[i0] - CELL/2, yv[j1] - CELL/2, xv[i1] + CELL/2, yv[j0] + CELL/2))
wlons, wlats = work_area.get_lonlats()
assert np.allclose(wlons, lons2d[j0:j1+1, i0:i1+1]) and \
       np.allclose(wlats, lats2d[j0:j1+1, i0:i1+1]), "crop is NOT cell-aligned!"
masks = {n: m[j0:j1+1, i0:i1+1] for n, m in masks_full.items()}
for n, m in masks.items():
    print(f"  {n}: {int(m.sum()):>6,} cells")
print(f"work window: {work_area.height} x {work_area.width} cells — alignment verified")

def list_months():
    with urllib.request.urlopen(f"{OISSS_MONTHLY}/", timeout=60) as r:
        html = r.read().decode("utf-8", "ignore")
    return sorted(set(re.findall(
        r"OISSS_L4_multimission_global_monthly_v3\.0_(\d{4}-\d{2})\.nc", html)))

MONTHS = list_months()
print(f"{len(MONTHS)} monthly files: {MONTHS[0]} -> {MONTHS[-1]}")

  AnadyrInner:  1,879 cells
  AnadyrOuter:  3,526 cells
  AnadyrStrait:  1,289 cells
  DBO1:  2,454 cells
  DBO2:    648 cells
  boxC:  3,480 cells
work window: 135 x 321 cells — alignment verified
176 monthly files: 2011-09 -> 2026-04


/srv/conda/envs/notebook/lib/python3.12/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


## 1. Build (or load) the EASE-2 monthly SSS stack
Same regrid recipe as the Yukon-side stack (gauss, 30 km radius, valid points only).
Source rows are lat 56–70 across the dateline (lon360 165–205), so the gulf and the DBO
line arrive on one continuous swath.

In [5]:
cache = OUT / "monthly_sss_ease2west_gauss.nc"
if cache.exists():
    stack = xr.open_dataarray(cache)
    print(f"cache: {dict(stack.sizes)} {str(stack.time.values[0])[:7]} -> "
          f"{str(stack.time.values[-1])[:7]}")
else:
    frames, times = [], []
    t0 = _time.time()
    for ym in MONTHS:
        url = f"{OISSS_MONTHLY}/OISSS_L4_multimission_global_monthly_v3.0_{ym}.nc"
        try:
            with urllib.request.urlopen(url, timeout=180) as r:
                raw = r.read()
            with xr.open_dataset(io.BytesIO(raw), engine="scipy") as ds:
                da = ds["sss"].squeeze()
                la = da["latitude"].values
                lat_sl = slice(56, 70) if la[0] < la[-1] else slice(70, 56)
                da = da.sel(latitude=lat_sl).load()
        except Exception as e:
            print(f"  skip {ym} ({type(e).__name__}: {str(e)[:40]})"); continue
        da = da.assign_coords(lon360=("longitude", da["longitude"].values % 360))
        da = da.swap_dims({"longitude": "lon360"}).sortby("lon360").sel(lon360=slice(165, 205))
        v = da.values.astype(float)
        v[(v <= SSS_VALID[0]) | (v > SSS_VALID[1])] = np.nan
        LO, LA = np.meshgrid(da.lon360.values, da.latitude.values)
        LO = np.where(LO > 180, LO - 360, LO)          # geodetic lons for the swath
        ok = np.isfinite(v)
        if ok.sum() == 0:
            print(f"  {ym}: empty, skipped"); continue
        sw = geometry.SwathDefinition(lons=LO[ok], lats=LA[ok])
        out = resample_gauss(sw, v[ok], work_area, radius_of_influence=ROI,
                             sigmas=ROI / 2, neighbours=8, fill_value=np.nan,
                             reduce_data=False)
        frames.append(out.astype(np.float32)); times.append(pd.Timestamp(ym + "-01"))
        if len(times) % 24 == 0:
            print(f"  {ym}: {len(times)} months ({_time.time()-t0:.0f}s)")
    stack = xr.DataArray(np.stack(frames), dims=("time", "y", "x"),
                         coords={"time": times}, name="sss")
    stack.to_netcdf(cache)
    print(f"built {dict(stack.sizes)} in {_time.time()-t0:.0f}s -> {cache}")

# sanity: region means must be finite for most months (gulf now inside the grid)
for n, m in masks.items():
    s = stack.where(xr.DataArray(m, dims=("y", "x"))).mean(("y", "x"), skipna=True)
    print(f"  {n}: {int(np.isfinite(s.values).sum())}/{stack.sizes['time']} months finite")

/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/

  2013-08: 24 months (8s)


/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/

  2015-08: 48 months (16s)


/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/

  2017-08: 72 months (23s)


... (6 outputs left out)

/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/

  2025-08: 168 months (54s)


/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyresample/kd_tree.py:261: UserWarning: Possible more than 8 neighbours within 30000 m for some data points
  get_neighbour_info(source_geo_def,
/srv/

built {'time': 176, 'y': 135, 'x': 321} in 57s -> oisss_west_cache/monthly_sss_ease2west_gauss.nc
  AnadyrInner: 121/176 months finite
  AnadyrOuter: 165/176 months finite
  AnadyrStrait: 117/176 months finite
  DBO1: 141/176 months finite
  DBO2: 103/176 months finite
  boxC: 111/176 months finite


---

# Part C — region means for boxes A/B/C and DBO 2

*This part was a separate notebook before the repository was consolidated; it starts from its own imports and configuration, so it can be run on its own after the sections above.*

# Boxes A/B/C + DBO 2 on the EASE-2 grid: trends & the box A ↔ DBO 2 adg correlation

Purpose: put boxes A/B/C on the **exact pipeline of the DBO trend notebooks**
(`1_pull_data/05_modis-kd490-8day.ipynb`, `1_pull_data/04_modis-adg443-8day.ipynb`) so grid/regridding is
eliminated as a difference, then quantify how adg anomalies co-vary between box A and DBO 2.

Faithful to the DBO notebooks (verbatim where possible):
* the same `dbo_area` **EASE-2 North LAEA 4 km** AreaDefinition (906×845; it already covers the
  Aleutians→Beaufort, so boxes A–C fit inside it),
* the same regrid: masked-swath `resample_nearest`, `radius_of_influence=6 km`, NaN fill,
* the same `v > 0` validity filter,
* the same canonical **DBO 2 polygon** and `matplotlib.path` masking (expect ~605 grid cells — the
  DBO notebooks printed 605; a small difference just reflects the cropped window edge),
* the same statistics: box-mean 8-day series → monthly means → month-of-year deseasonalization →
  Theil–Sen + AR(1)-corrected Mann–Kendall.

Different on purpose:
* **Reads from `bering_full_cache/`** (the full-Bering notebook's granule cache) instead of
  re-streaming from Earthdata — nothing is downloaded. Run this where that cache lives.
* Resamples into a **cropped, cell-aligned window** of `dbo_area` (verified by assertion) so the
  per-composite regrid loop takes ~10–25 min total instead of hours.

Reference numbers to compare against (established earlier):
* DBO notebooks, DBO 2, monthly, record through 2025-11-17: **adg +0.0072/dec (p=0.022*)**,
  **Kd +0.0254/dec (p=0.008**)**, n=150.
* Native-grid (no EASE regrid) equivalents at the same window: adg +0.0058* / Kd +0.0110 ns.

In [2]:
import pickle, time as _time, warnings
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from pyresample import AreaDefinition, geometry
from pyresample.kd_tree import resample_nearest
from matplotlib.path import Path as MplPath
from scipy.stats import theilslopes, kendalltau, norm, t as tdist

warnings.filterwarnings("ignore", category=RuntimeWarning)

## 1. Configuration
`dbo_area` is copied **verbatim** from the DBO trend notebooks. `CACHE` must point at the
full-Bering notebook's granule cache (0–360 longitudes).

In [3]:
# --- EASE-2 grid: verbatim from 1_pull_data/05_modis-kd490-8day.ipynb ---------------
dbo_area = AreaDefinition(
    area_id="ease2_north_dbo_4km",
    description="EASE-Grid 2.0 North subset: Aleutians -> Beaufort (DBO1-8), 4 km",
    proj_id="EPSG:6931",
    projection={"proj": "laea", "lat_0": 90, "lon_0": 0, "x_0": 0, "y_0": 0,
                "datum": "WGS84", "units": "m"},
    width=906, height=845, area_extent=(-3_624_000, 992_000, 0, 4_372_000))
ROI = 6_000                       # nearest-neighbour radius (m); L3m is 4 km

# canonical DBO 2 polygon (the DBO weekly-pipeline box definitions (not included), (lat, lon) corners)
DBO2_RING = [(65.107747, -170.865176), (65.097001, -167.859549),
             (64.481828, -167.907758), (64.493658, -170.840669)]

# study boxes (rectangles, same as the box / full-Bering notebooks)
BOXES = {
    "A": dict(lat=(57.7, 59.7), lon=(-171.5, -166.5)),
    "B": dict(lat=(60.2, 62.2), lon=(-170.5, -165.5)),
    "C": dict(lat=(62.4, 64.4), lon=(-169.0, -164.0)),
}
DBO2E_SPLIT = -169.3              # east of this = ACC / Yukon-corridor side of DBO 2

PRODUCTS = {
    "adg443": dict(name="adg_443 (GIOP)"),
    "kd490":  dict(name="Kd_490"),
}
THEIR_WINDOW_END = "2025-11-30"   # the DBO notebooks' record ended 2025-11-17
FULL_END         = None           # None = everything in the cache

CACHE  = Path("bering_full_cache")            # <- the full-Bering notebook's cache
OUTDIR = Path("ease2_output"); OUTDIR.mkdir(exist_ok=True)
assert CACHE.exists(), "bering_full_cache/ not found — run this next to the full-Bering notebook"

## 2. Region masks on the EASE-2 grid, and the cropped work window

Masks are built with the DBO notebooks' method (polygon `contains_points` for DBO 2; lat/lon bounds
for the rectangles) on the **full** `dbo_area`, then everything is cropped to the smallest cell-
aligned window containing all regions. The assertion proves the cropped cells are identical to the
corresponding cells of the full grid — i.e. we are still on *their* grid, just not resampling the
Beaufort for nothing.

In [4]:
lons2d, lats2d = dbo_area.get_lonlats()
grid_pts = np.column_stack([lons2d.ravel(), lats2d.ravel()])

masks = {}
ring = [(lo, la) for la, lo in DBO2_RING]
masks["DBO2"] = MplPath(ring + [ring[0]]).contains_points(grid_pts).reshape(lons2d.shape)
masks["DBO2east"] = masks["DBO2"] & (lons2d >= DBO2E_SPLIT)
for bname, b in BOXES.items():
    masks[bname] = ((lats2d >= b["lat"][0]) & (lats2d <= b["lat"][1]) &
                    (lons2d >= b["lon"][0]) & (lons2d <= b["lon"][1]))
for name, m in masks.items():
    print(f"  {name}: {int(m.sum()):>6,} grid cells")   # DBO2 should print ~605

# --- crop to a cell-aligned sub-window covering all regions (+pad) ----------------------
union = np.zeros_like(masks["DBO2"])
for m in masks.values():
    union |= m
jj, ii = np.where(union)
PAD = 5
j0, j1 = max(jj.min() - PAD, 0), min(jj.max() + PAD, dbo_area.height - 1)
i0, i1 = max(ii.min() - PAD, 0), min(ii.max() + PAD, dbo_area.width - 1)

xv, yv = dbo_area.get_proj_vectors()          # cell centers; y decreasing (origin upper)
cell = 4000.0
work_area = AreaDefinition(
    area_id="ease2_work", description="cell-aligned crop of dbo_area", proj_id="EPSG:6931",
    projection=dbo_area.crs.to_dict() if hasattr(dbo_area, "crs") else dbo_area.proj_dict,
    width=i1 - i0 + 1, height=j1 - j0 + 1,
    area_extent=(xv[i0] - cell / 2, yv[j1] - cell / 2, xv[i1] + cell / 2, yv[j0] + cell / 2))

wlons, wlats = work_area.get_lonlats()
assert np.allclose(wlons, lons2d[j0:j1 + 1, i0:i1 + 1]) and        np.allclose(wlats, lats2d[j0:j1 + 1, i0:i1 + 1]), "crop is NOT cell-aligned!"
masks = {n: m[j0:j1 + 1, i0:i1 + 1] for n, m in masks.items()}
print(f"work window: {work_area.height} x {work_area.width} cells "
      f"(rows {j0}-{j1}, cols {i0}-{i1} of dbo_area) — alignment verified")

fig, ax = plt.subplots(figsize=(7, 6))
canvas = np.zeros(wlons.shape)
for k, (n, m) in enumerate(masks.items(), start=1):
    canvas[m] = k
ax.imshow(canvas, origin="upper", interpolation="nearest", cmap="tab10")
ax.set_title("region masks on the EASE-2 work window (sanity check)")
plt.show()

  DBO2:    605 grid cells
  DBO2east:    289 grid cells
  A:  4,034 grid cells
  B:  3,743 grid cells
  C:  3,480 grid cells
work window: 215 x 111 cells (rows 209-423, cols 691-801 of dbo_area) — alignment verified


/srv/conda/envs/notebook/lib/python3.12/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


## 3. Regrid every cached composite to EASE-2 and take region means

Faithful to the DBO notebooks' `grid_to_area`: per composite, keep only finite `v > 0` pixels,
build a swath from those, `resample_nearest` with ROI 6 km (so a cloudy cell borrows a valid
neighbour up to 6 km away, exactly as their fields did). Checkpointed — safe to interrupt/re-run.
~10–25 min for both products.

In [5]:
def granule_times(fname):
    d0, d1 = fname.split(".")[1].split("_")
    t0, t1 = pd.Timestamp(d0), pd.Timestamp(d1)
    return t0, t1, t0 + (t1 - t0) / 2

def region_means_for_file(f, key):
    """One cached composite -> {region: (mean, n_valid_cells)} on the EASE-2 work window."""
    with xr.open_dataset(f, engine="h5netcdf") as ds:
        v = np.asarray(ds[key].values, float).squeeze()
        la = np.asarray(ds.lat.values, float)
        lo = np.asarray(ds.lon.values, float)
    lo = np.where(lo > 180.0, lo - 360.0, lo)          # cache is 0-360; swath wants -180..180
    v = np.where(v > 0, v, np.nan)                     # their validity filter, verbatim
    lon2d, lat2d = np.meshgrid(lo, la)
    m = np.isfinite(v)
    if not m.any():
        arr = np.full(work_area.shape, np.nan, np.float32)
    else:
        swath = geometry.SwathDefinition(lons=lon2d[m], lats=lat2d[m])
        arr = resample_nearest(swath, v[m], work_area, radius_of_influence=ROI,
                               fill_value=np.nan, reduce_data=False).astype(np.float32)
    out = {}
    for name, mask in masks.items():
        vals = arr[mask]
        vals = vals[np.isfinite(vals)]
        out[name] = (float(vals.mean()) if vals.size else np.nan, int(vals.size))
    return out

CKPT = OUTDIR / "ease2_region_means_ckpt.pkl"
rows = pickle.load(open(CKPT, "rb")) if CKPT.exists() else {k: {} for k in PRODUCTS}

for key in PRODUCTS:
    files = sorted((CACHE / key).glob("*.nc"))
    todo = [f for f in files if f.name not in rows[key]]
    print(f"{key}: {len(files)} composites, {len(todo)} to do")
    t0 = _time.time()
    for i, f in enumerate(todo):
        try:
            rows[key][f.name] = region_means_for_file(f, key)
        except Exception as ex:
            print(f"  FAIL {f.name}: {str(ex)[:70]}")
        if (i + 1) % 200 == 0:
            pickle.dump(rows, open(CKPT, "wb"))
            rate = (i + 1) / (_time.time() - t0)
            print(f"  {i+1}/{len(todo)}  ({(len(todo)-i-1)/rate/60:.1f} min left)")
    pickle.dump(rows, open(CKPT, "wb"))
    print(f"  {key} done in {(_time.time()-t0)/60:.1f} min")

# tidy into per-(product, region) time series
means, counts = {}, {}
for key in PRODUCTS:
    times = [granule_times(fn)[2] for fn in sorted(rows[key])]
    for name in masks:
        vals = [rows[key][fn][name][0] for fn in sorted(rows[key])]
        cnts = [rows[key][fn][name][1] for fn in sorted(rows[key])]
        means[(key, name)]  = pd.Series(vals, index=pd.DatetimeIndex(times)).sort_index()
        counts[(key, name)] = pd.Series(cnts, index=pd.DatetimeIndex(times)).sort_index()
print("series built:", list(means)[:4], "...")

adg443: 1099 composites, 0 to do
  adg443 done in 0.0 min
kd490: 1099 composites, 0 to do
  kd490 done in 0.0 min
series built: [('adg443', 'DBO2'), ('adg443', 'DBO2east'), ('adg443', 'A'), ('adg443', 'B')] ...


## 4. Trends, the DBO-notebook way
Box-mean 8-day series → monthly mean → month-of-year anomaly → Theil–Sen + AR(1)-MK.
Two windows: the DBO notebooks' record end (**2025-11**, the apples-to-apples one) and the full
current record. Compare the DBO2 rows with the reference numbers in the header.

In [6]:
# --- trend recipe (the DBO weekly-pipeline trend recipe (not included)) ------------------
def _lag1(r):
    r = np.asarray(r, float)
    a, b = r[1:], r[:-1]
    v = np.isfinite(a) & np.isfinite(b)
    return float(np.corrcoef(a[v], b[v])[0, 1]) if v.sum() >= 10 else 0.0

def theilsen_mk_ar1(x, y, minn=8):
    v = np.isfinite(x) & np.isfinite(y)
    x, y = np.asarray(x, float)[v], np.asarray(y, float)[v]
    n = len(x)
    if n < minn:
        return dict(slope=np.nan, intercept=np.nan, tau=np.nan,
                    p_naive=np.nan, p_adj=np.nan, phi=np.nan, n=n)
    slope, intercept, _, _ = theilslopes(y, x)
    tau, p_naive = kendalltau(x, y)
    sl, ic = np.polyfit(x, y, 1)
    phi = _lag1(y - (sl * x + ic))
    n_eff = n * (1 - phi) / (1 + phi) if (abs(phi) < 1 and phi != 0) else n
    p_adj = (2 * (1 - norm.cdf(abs(tau / np.sqrt(2 * (2 * n_eff + 5) / (9 * n_eff * (n_eff - 1))))))
             if n_eff > 4 else np.nan)
    return dict(slope=float(slope), intercept=float(intercept), tau=float(tau),
                p_naive=float(p_naive), p_adj=float(p_adj), phi=float(phi), n=n)

def stars(p):
    if p is None or not np.isfinite(p):
        return "n/a"
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
# -----------------------------------------------------------------------------------------

def monthly_anom(key, region, end=None):
    """Monthly means of the 8-day box series, minus month-of-year climatology."""
    s = means[(key, region)]
    if end is not None:
        s = s[:end]
    m = s.resample("MS").mean()
    clim = m.groupby(m.index.month).mean()
    anom = m - m.index.month.map(clim)
    dec = np.asarray(m.index.year + (m.index.month - 0.5) / 12.0, float)
    return dec, anom

REGION_ORDER = ["A", "B", "C", "DBO2", "DBO2east"]
tbl = []
for key in PRODUCTS:
    for region in REGION_ORDER:
        for wname, end in [("thru 2025-11 (their window)", THEIR_WINDOW_END),
                           ("full record", FULL_END)]:
            dec, anom = monthly_anom(key, region, end)
            r = theilsen_mk_ar1(dec, anom.values)
            base = float(np.nanmean(means[(key, region)]))
            tbl.append(dict(product=PRODUCTS[key]["name"], region=region, window=wname,
                            slope_per_decade=r["slope"] * 10,
                            pct_per_decade=100 * r["slope"] * 10 / base if base else np.nan,
                            p_adj=r["p_adj"], sig=stars(r["p_adj"]), n_months=r["n"]))
trends_tbl = pd.DataFrame(tbl)
trends_tbl.to_csv(OUTDIR / "ease2_trend_table.csv", index=False)
print("reference: DBO nbs DBO2 = adg +0.0072* / Kd +0.0254** ; "
      "native grid @ their window = adg +0.0058* / Kd +0.0110 ns\n")
trends_tbl.round(4)

reference: DBO nbs DBO2 = adg +0.0072* / Kd +0.0254** ; native grid @ their window = adg +0.0058* / Kd +0.0110 ns



,product,region,window,slope_per_decade,pct_per_decade,p_adj,sig,n_months
0,adg_443 (GIOP),A,thru 2025-11 (their window),0.0006,1.2271,0.8262,ns,206
1,adg_443 (GIOP),A,full record,0.0010,2.2889,0.6809,ns,208
2,adg_443 (GIOP),B,thru 2025-11 (their window),0.0048,7.5146,0.1304,ns,194
3,adg_443 (GIOP),B,full record,0.0044,6.8889,0.1637,ns,196
4,adg_443 (GIOP),C,thru 2025-11 (their window),0.0062,8.3221,0.1606,ns,187
5,adg_443 (GIOP),C,full record,0.0050,6.7948,0.2293,ns,190
6,adg_443 (GIOP),DBO2,thru 2025-11 (their window),0.0052,9.6865,0.0530,ns,168
7,adg_443 (GIOP),DBO2,full record,0.0036,6.7027,0.1548,ns,171
8,adg_443 (GIOP),DBO2east,thru 2025-11 (their window),0.0063,12.2431,0.0850,ns,160
9,adg_443 (GIOP),DBO2east,full record,0.0050,9.7905,0.1625,ns,162
